In [7]:

from abc import ABC, abstractmethod

# Product
class Animal(ABC):
    @abstractmethod
    def speak(self):
        pass

class Dog(Animal):
    def __init__(self, name="Dog"):
        self.name = name

    def speak(self):
        return f"{self.name} says Woof!"

class Cat(Animal):
    def __init__(self, name="Cat"):
        self.name = name

    def speak(self):
        return f"{self.name} says Meow!"

# Factory
class AnimalFactory:
    def create_animal(self, config):
        # config is expected to be a dict like {"type": "Dog", "name": "Fido"}
        if not isinstance(config, dict) or "type" not in config:
            raise ValueError("Config dictionary must have a 'type' key.")
        class_name = config["type"]
        # Remove the 'type' key before passing the rest as kwargs
        kwargs = {k: v for k, v in config.items() if k != "type"}
        animal_class = eval(class_name)
        return animal_class(**kwargs)  # Pass possible constructor arguments

# Usage
factory = AnimalFactory()
animal1 = factory.create_animal({"type": "Dog", "name": "Rex"})
animal2 = factory.create_animal({"type": "Cat", "name": "Whiskers"})

print(animal1.speak())  # Output: Rex says Woof!
print(animal2.speak())  # Output: Whiskers says Meow!



Rex says Woof!
Whiskers says Meow!


In [8]:
animal1 = factory.create_animal({"type": "Tiger"})

NameError: name 'Tiger' is not defined

In [9]:

# Define Planning classes
class MonolithicPlan:
    def __init__(self, description="Monolithic Planning"):
        self.description = description

    def run(self):
        return f"Executing {self.description}."

class SpatialDecomposedPlan:
    def __init__(self, description="Spatial Decomposed Planning"):
        self.description = description

    def run(self):
        return f"Executing {self.description}."

# Generic Factory for planning
class PlanningFactory:
    def create_plan(self, config):
        if not isinstance(config, dict) or "type" not in config:
            raise ValueError("Config dictionary must have a 'type' key.")
        class_name = config["type"]
        kwargs = {k: v for k, v in config.items() if k != "type"}
        plan_class = eval(class_name)
        return plan_class(**kwargs)

# Usage
plan_factory = PlanningFactory()
plan1 = plan_factory.create_plan({"type": "MonolithicPlan", "description": "Unified Approach"})
plan2 = plan_factory.create_plan({"type": "SpatialDecomposedPlan", "description": "Divide and Conquer"})

print(plan1.run())  # Output: Executing Unified Approach.
print(plan2.run())  # Output: Executing Divide and Conquer.




Executing Unified Approach.
Executing Divide and Conquer.


In [10]:
class TemporalDecomposedPlan:
    def __init__(self, description="Temporal Decomposed Planning"):
        self.description = description

    def run(self):
        return f"Executing {self.description}."

# Create an instance of the new plan class using the PlanningFactory
plan3 = plan_factory.create_plan({"type": "TemporalDecomposedPlan", "description": "Time-Based Separation"})
print(plan3.run())  # Output: Executing Time-Based Separation.


Executing Time-Based Separation.


In [1]:
# A good solution for this is the Decorator pattern, which allows you to dynamically add responsibilities (in this case, decompositions) to objects without class explosion.

from abc import ABC, abstractmethod

# Define an abstract Plan interface
class Plan(ABC):
    @abstractmethod
    def run(self):
        pass

# Concrete Plan implementation
class MonolithicPlan(Plan):
    def __init__(self, description="Unified Approach"):
        self.description = description

    def run(self):
        return f"Executing {self.description}."

# Decorator base class
class PlanDecorator(Plan):
    def __init__(self, plan):
        self.plan = plan

    def run(self):
        return self.plan.run()

# Spatial decomposition decorator
class SpatialDecompositionDecorator(PlanDecorator):
    def run(self):
        return f"{self.plan.run()} [with Spatial Decomposition]"

# Temporal decomposition decorator
class TemporalDecompositionDecorator(PlanDecorator):
    def run(self):
        return f"{self.plan.run()} [with Temporal Decomposition]"

# Example usage:
base_plan = MonolithicPlan("My Project")
spatial_plan = SpatialDecompositionDecorator(base_plan)
temporal_plan = TemporalDecompositionDecorator(base_plan)
both_decomposed = TemporalDecompositionDecorator(SpatialDecompositionDecorator(base_plan))

print(base_plan.run())           # Executing My Project.
print(spatial_plan.run())        # Executing My Project. [with Spatial Decomposition]
print(temporal_plan.run())       # Executing My Project. [with Temporal Decomposition]
print(both_decomposed.run())     # Executing My Project. [with Spatial Decomposition] [with Temporal Decomposition]



Executing My Project.
Executing My Project. [with Spatial Decomposition]
Executing My Project. [with Temporal Decomposition]
Executing My Project. [with Spatial Decomposition] [with Temporal Decomposition]


In [1]:
# Example: Simple Factory Pattern for an ML system using PyTorch

import torch
import torch.nn as nn

# Abstract Product
class BaseModel(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        raise NotImplementedError

# Concrete Products
class SimpleMLP(BaseModel):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size)
        )

    def forward(self, x):
        return self.model(x)

class SimpleCNN(BaseModel):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 8, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(8*28*28, num_classes)
        )

    def forward(self, x):
        return self.conv(x)

# Factory
class ModelFactory:
    def create_model(self, config):
        model_type = config.get("type")
        if model_type == "mlp":
            return SimpleMLP(
                input_size=config["input_size"],
                hidden_size=config["hidden_size"],
                output_size=config["output_size"]
            )
        elif model_type == "cnn":
            return SimpleCNN(
                in_channels=config["in_channels"],
                num_classes=config["num_classes"]
            )
        else:
            raise ValueError(f"Unknown model type: {model_type}")

# Usage Example:
factory = ModelFactory()
mlp_config = {"type": "mlp", "input_size": 10, "hidden_size": 20, "output_size": 2}
cnn_config = {"type": "cnn", "in_channels": 1, "num_classes": 10}

mlp = factory.create_model(mlp_config)
cnn = factory.create_model(cnn_config)

x_mlp = torch.randn(5, 10)         # (batch, input_size)
x_cnn = torch.randn(5, 1, 28, 28)  # (batch, channels, height, width)

print(mlp(x_mlp))
print(cnn(x_cnn))


tensor([[ 0.0651, -0.0088],
        [ 0.0941,  0.1705],
        [ 0.0190,  0.3744],
        [ 0.2190,  0.1178],
        [-0.1613,  0.1767]], grad_fn=<AddmmBackward0>)
tensor([[-0.0525, -0.1999, -0.1536,  0.0102, -0.0245,  0.2797, -0.0659, -0.1945,
         -0.0733, -0.0553],
        [-0.3686, -0.2315,  0.0850, -0.3440, -0.0357,  0.2565,  0.0340,  0.1119,
         -0.1763,  0.2947],
        [-0.3279,  0.0493, -0.3442,  0.0128, -0.2121,  0.1835, -0.2913,  0.0993,
         -0.3600,  0.1523],
        [ 0.0019, -0.3381, -0.4004, -0.2387, -0.0403,  0.4034,  0.0296,  0.1064,
         -0.3828,  0.0488],
        [ 0.0394, -0.0357, -0.0673, -0.0917, -0.1687,  0.1809,  0.0008, -0.0346,
         -0.0452,  0.0700]], grad_fn=<AddmmBackward0>)
